# Privacy Guardian — Colab GPU Worker

Optional temporary compute runtime for **approved heavy work only**
(Ollama reasoning, vision/OCR parsing, embeddings). The laptop remains the
**system of record**. Colab is disposable: it never stores the evidence
database and never decides results are `completed` on its own behalf.

> Privacy gates (non-negotiable): LOCAL ONLY mode never sends data here;
> HYBRID requires explicit user approval; no credentials in payloads; temp
> sensitive data is deleted before the final acknowledgement.

Workflow: setup -> advertise capabilities -> poll dispatcher -> process ->
delete temp data -> return structured result. If this VM vanishes mid-job the
laptop marks the job `interrupted` (never auto-`completed`).

## 1 · Setup

Run the next cell to make the `colab/` package importable.

Two options:
- **A.** Clone this repository (public URL) into `/content/privacy-guardian`. Fastest.
- **B.** Upload the `colab/` folder (File > Upload) into `/content` and skip cloning.

Set `COLAB_JOB_DISPATCHER_URL` below to the **laptop dispatcher** endpoint
(exposed via a tunnel such as loca.lt), e.g. `https://xxx.loca.lt`. Leave
empty to run a self-check only.

In [ ]:
# ===== CONFIGURATION =====
REPO_URL = "https://github.com/S-Q-Ali/Mis-Clear.git"   # public repo (Option A)
COLAB_JOB_DISPATCHER_URL = ""    # laptop dispatcher tunnel, e.g. https://xxx.loca.lt
MAX_ITERATIONS = 30              # poll-and-process iterations (worker loop)
POLL_INTERVAL_S = 5              # seconds between polls
PROBE_GPU = True                 # advertise GPU capabilities (never required)
RUN_DIR = "/content/privacy-guardian"


In [ ]:
import os, sys
from pathlib import Path

run_dir = Path(RUN_DIR)
if not (run_dir / "colab").exists() and REPO_URL.startswith("http"):
    print("cloning repo (option A)...")
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(run_dir)], check=True)
sys.path.insert(0, str(run_dir) if (run_dir / "colab").exists() else "/content")
import colab.worker as worker
print("colab package importable:", "OK")


### Optional GPU check
The worker always works on CPU; the GPU probe only advertises extra
capabilities (`gpu_count`, `gpus[]`, `vidMemory`). Everything remains
synthetic-safe: no model is downloaded unless you explicitly opt in.

In [ ]:
from colab.capabilities import detect_capabilities

caps = detect_capabilities(probe_gpu=PROBE_GPU)
print("capabilities advertised:")
for k, v in caps.advertise().items():
    print(f"  {k}: {v}")


In [ ]:
# === self-check: worker badge without a dispatcher ===
from colab.protocol import JobRequest
from colab.capabilities import CapabilityReport

fake = JobRequest.from_dict({
    "protocol_version": "1", "job_id": "nb-selfcheck",
    "job_type": "reasoning", "created_at": "", "expires_at": "",
    "privacy_mode": "hybrid_approved", "payload": {"text": "hello"},
    "requested_capabilities": [], "return_format": "json"})
res, detail = worker.execute_job(
    fake, caps=CapabilityReport(cpu=True, models=[]),
    tmp_root="/content/data/cache/colab")
print(f"self-check status={res.status} data_deleted={res.data_deleted}")
if res.errors:
    print("honest errors:", res.errors)
assert res.data_deleted is True


In [ ]:
# ==== main worker loop (approval-gated) ====
if not COLAB_JOB_DISPATCHER_URL:
    print("COLAB_JOB_DISPATCHER_URL is empty; skipping dispatch loop (self-check only).")
else:
    summary = worker.run_worker_main(
        COLAB_JOB_DISPATCHER_URL,
        caps=caps,
        tmp_root="/content/data/cache/colab",
        max_iterations=MAX_ITERATIONS,
        poll_interval_s=POLL_INTERVAL_S,
    )
    print("summary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")
    if summary.get("note"):
        print("note:", summary["note"])


## Cleanup & honesty

- `data/cache/colab/<job_id>` temporary folders are removed before each result
  is acknowledged (`data_deleted: true`).
- If the dispatcher is unreachable the worker retries and then stops with an
  honest summary (`reconnects`, `note`) — it never fabricates a completed job.
- Stop this runtime when done.

## Next
Set `PG_COLAB_JOB_DISPATCHER_URL` in your laptop `.env` to the same tunnel URL,
create a job via `POST /api/jobs`, then `POST /api/jobs/{id}/dispatch` and
`POST /api/jobs/{id}/poll`. See docs/COLAB_GPU_ARCHITECTURE.md.